# Assignment 04 - Project: Deep Research with LangGraph - Research Agent with MCP
This notebook implements **Module 4: Research Agent with MCP - Research Agent with MCP**. We simulate a Model Context Protocol (MCP) server that exposes database search and file reading resources, and build a research agent that queries it.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
from mock_llm import get_llm

load_dotenv()
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
class MockMCPServer:
    """Simulated Model Context Protocol (MCP) Server exposing tools & resources."""
    def __init__(self):
        self.resources = {
            "papers/post_quantum_standards.txt": "NIST finalized standard definitions for lattice-based algorithms in 2024.",
            "papers/quantum_timelines.txt": "Practical decryption capabilities are projected by researchers for 2030-2035."
        }
        
    def list_resources(self) -> list:
        return list(self.resources.keys())
        
    def read_resource(self, path: str) -> str:
        return self.resources.get(path, "Error: Resource not found.")
        
    def search_database(self, query: str) -> str:
        return f"[MCP Server Search for '{query}']: Found lattice standardizations finalized in FIPS 203."

In [3]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END

class MCPState(TypedDict):
    brief: str
    available_resources: List[str]
    target_resource: str
    read_contents: str
    notes: str

In [4]:
mcp_server = MockMCPServer()

def discover_mcp_resources(state: MCPState):
    """Node: Query the MCP server to list available files/resources."""
    resources = mcp_server.list_resources()
    return {"available_resources": resources}

In [5]:
def selector_mcp_resource(state: MCPState):
    """Node: Decide which MCP resource is relevant based on our brief."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Brief: {state['brief']}
Available MCP Files:
{state['available_resources']}
Which single file should we read? Return the file path only."""
    
    res = llm.invoke(prompt)
    # Simple validation
    chosen = res.content.strip().strip("'").strip('"')
    if chosen not in state["available_resources"]:
        chosen = state["available_resources"][0] # Fallback
    return {"target_resource": chosen}

In [6]:
def execute_mcp_read(state: MCPState):
    """Node: Fetch file contents from the MCP server."""
    contents = mcp_server.read_resource(state["target_resource"])
    return {"read_contents": contents}

In [7]:
def synthesize_mcp_findings(state: MCPState):
    """Node: Compile findings into report notes."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Brief: {state['brief']}
MCP File Contents Read from '{state['target_resource']}':
{state['read_contents']}
Synthesize this into a structured final report notes."""
    
    res = llm.invoke(prompt)
    return {"notes": res.content}

In [8]:
# Compile the MCP graph
builder = StateGraph(MCPState)
builder.add_node("discover", discover_mcp_resources)
builder.add_node("select", selector_mcp_resource)
builder.add_node("read", execute_mcp_read)
builder.add_node("synthesize", synthesize_mcp_findings)

builder.add_edge(START, "discover")
builder.add_edge("discover", "select")
builder.add_edge("select", "read")
builder.add_edge("read", "synthesize")
builder.add_edge("synthesize", END)

graph = builder.compile()

In [9]:
# Print the graph architecture
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("Could not draw graph:", e)

+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
 +----------+  
 | discover |  
 +----------+  
       *       
       *       
       *       
  +--------+   
  | select |   
  +--------+   
       *       
       *       
       *       
   +------+    
   | read |    
   +------+    
       *       
       *       
       *       
+------------+ 
| synthesize | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


In [10]:
initial_state = {
    "brief": "Researching finalized lattice standards.",
    "available_resources": [],
    "target_resource": "",
    "read_contents": "",
    "notes": ""
}

print("--- Executing MCP Research Agent ---")
res = graph.invoke(initial_state)
print(f"\nFetched Resource: {res['target_resource']}")
print("\n--- Synthesized Notes ---")
print(res["notes"])

--- Executing MCP Research Agent ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



Fetched Resource: papers/post_quantum_standards.txt

--- Synthesized Notes ---
**NIST 2024 Lattice‑Based Post‑Quantum Cryptography (PQC) Standards – Final Report Notes**  
*(Synthesised from the “papers/post_quantum_standards.txt” file containing the NIST final‑standard definitions for lattice‑based algorithms)*  

---

## 1. Executive Summary
- **Scope**: Consolidates the NIST‑approved lattice‑based cryptographic primitives that were finalized in the 2024 PQC standardisation round.  
- **Algorithms Covered**:  
  1. **CRYSTALS‑Kyber** – Key‑Encapsulation Mechanism (KEM)  
  2. **CRYSTALS‑Dilithium** – Digital Signature Scheme  
  3. **Falcon** – Digital Signature Scheme (alternative, not mandatory)  

- **Security Levels**: All three algorithms provide three NIST security “strengths” – **Level 1**, **Level 3**, and **Level 5** – mapping to roughly RSA‑2048, RSA‑3072, and RSA‑15360 security, respectively.  
- **Key Take‑aways**:  
  - Kyber and Dilithium are the **mandatory** algorith